<a href="https://colab.research.google.com/github/artstudio-mayer/Finanzen/blob/main/Zuschnitt_Leisten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code für die Berechnung des optimalen Zuschnitts

In [40]:
# Länge der eingekauften Leisten
stamm_laenge = 240

# Bedarfe cm: #
bedarfe = {
    142: 2,
    117: 4,
    77: 6,
    72: 4,
    57: 2,
    52: 18}

In [42]:
def loese_zuschnitt_mit_maximalem_endrest():

    # 1. Generiere alle mathematisch gültigen Muster unter 240cm
    einzelne_teile = sorted(list(bedarfe.keys()), reverse=True)
    gueltige_muster = []

    def finde_muster(index, aktuelles_muster, rest_laenge):
        if rest_laenge >= 0:
            gueltige_muster.append(list(aktuelles_muster))
        if index == len(einzelne_teile) or rest_laenge <= 0:
            return

        teil = einzelne_teile[index]
        max_anzahl = min(bedarfe[teil], rest_laenge // teil)
        for anzahl in range(max_anzahl, -1, -1):
            for _ in range(anzahl):
                aktuelles_muster.append(teil)
            finde_muster(index + 1, aktuelles_muster, rest_laenge - anzahl * teil)
            for _ in range(anzahl):
                aktuelles_muster.pop()

    finde_muster(0, [], stamm_laenge)

    # Sortiere Muster: Maximale Materialbelegung (wenig Verschnitt) nach vorne
    gueltige_muster.sort(key=lambda m: (stamm_laenge - sum(m), len(m)))

    # 2. Muster-Auswahl mit Sonderlogik für die letzte Leiste
    rest_bedarf = bedarfe.copy()
    gewaehlte_leisten = []

    while sum(rest_bedarf.values()) > 0:
        # Ermittle alle aktuell noch benötigten Einzelteile
        verbleibende_teile = []
        for teil, anzahl in rest_bedarf.items():
            verbleibende_teile.extend([teil] * anzahl)

        # BEDINGUNG: Passen alle restlichen Teile auf eine einzige Leiste?
        if sum(verbleibende_teile) <= stamm_laenge:
            # Ja! Das ist unsere letzte Leiste. Wir packen sie nicht weiter an.
            gewaehlte_leisten.append(verbleibende_teile)
            break

        # Ansonsten: Wähle das Muster, das die aktuelle Leiste am besten füllt
        bestes_muster = None
        beste_wertung = -1

        for muster in gueltige_muster:
            temp_bedarf = rest_bedarf.copy()
            gueltig = True
            nutzen = 0

            for teil in muster:
                if temp_bedarf[teil] > 0:
                    temp_bedarf[teil] -= 1  # Hier war der Tippfehler korrigiert
                    nutzen += teil
                else:
                    gueltig = False
                    break

            if gueltig and nutzen > beste_wertung:
                beste_wertung = nutzen
                bestes_muster = muster

        # Bedarf aktualisieren
        for teil in bestes_muster:
            rest_bedarf[teil] -= 1
        gewaehlte_leisten.append(bestes_muster)

    # 3. Optimierte Ausgabe
    print(f"=== ZUSCHNITTPLAN MIT MAXIMALEM END-REST ===")
    print(f"Benötigte 240cm-Leisten gesamt: {len(gewaehlte_leisten)}\n")

    for i, leiste in enumerate(gewaehlte_leisten):
        belegt = sum(leiste)
        verschnitt = stamm_laenge - belegt

        # Markiere die letzte Leiste optisch
        if i == len(gewaehlte_leisten) - 1:
            print(f"Leiste {i+1:2d}: Schnitte -> {sorted(leiste, reverse=True)} | Belegt: {belegt}cm | ⭐ NUTZBARER REST: {verschnitt}cm ⭐")
        else:
            print(f"Leiste {i+1:2d}: Schnitte -> {sorted(leiste, reverse=True)} | Belegt: {belegt}cm | Verschnitt: {verschnitt}cm")

if __name__ == "__main__":
    loese_zuschnitt_mit_maximalem_endrest()


=== ZUSCHNITTPLAN MIT MAXIMALEM END-REST ===
Benötigte 240cm-Leisten gesamt: 12

Leiste  1: Schnitte -> [77, 57, 52, 52] | Belegt: 238cm | Verschnitt: 2cm
Leiste  2: Schnitte -> [77, 57, 52, 52] | Belegt: 238cm | Verschnitt: 2cm
Leiste  3: Schnitte -> [117, 117] | Belegt: 234cm | Verschnitt: 6cm
Leiste  4: Schnitte -> [117, 117] | Belegt: 234cm | Verschnitt: 6cm
Leiste  5: Schnitte -> [77, 52, 52, 52] | Belegt: 233cm | Verschnitt: 7cm
Leiste  6: Schnitte -> [77, 52, 52, 52] | Belegt: 233cm | Verschnitt: 7cm
Leiste  7: Schnitte -> [77, 52, 52, 52] | Belegt: 233cm | Verschnitt: 7cm
Leiste  8: Schnitte -> [77, 52, 52, 52] | Belegt: 233cm | Verschnitt: 7cm
Leiste  9: Schnitte -> [72, 72, 72] | Belegt: 216cm | Verschnitt: 24cm
Leiste 10: Schnitte -> [142, 72] | Belegt: 214cm | Verschnitt: 26cm
Leiste 11: Schnitte -> [142, 52] | Belegt: 194cm | Verschnitt: 46cm
Leiste 12: Schnitte -> [52] | Belegt: 52cm | ⭐ NUTZBARER REST: 188cm ⭐
